In [22]:
import os
import re

TARGET_PDF = "test_files/Grand Theft Auto VI Trailer 1_recovered.pdf"

if not os.path.exists(TARGET_PDF):
  print(f"File not found: {TARGET_PDF}")
else:
  with open(TARGET_PDF, "rb") as f:
    data = f.read()

  file_size = len(data)
  print(f"Target: {os.path.basename(TARGET_PDF)} ({file_size:,} bytes)")
  print("-" * 60)

  # Check overlay / appended data after final %%EOF
  eof_positions = [m.start() for m in re.finditer(b"%%EOF", data)]
  if not eof_positions:
    print(
        "[!] ALERT: No '%%EOF' marker found. PDF may be truncated or corrupted."
    )
  else:
    last_eof = max(eof_positions)
    end_of_marker = last_eof + 5
    while end_of_marker < file_size and data[
        end_of_marker : end_of_marker + 1
    ] in (b"\r", b"\n", b" ", b"\t"):
      end_of_marker += 1

    trailing_len = file_size - end_of_marker
    if trailing_len > 0:
      trailing_preview = data[end_of_marker : end_of_marker + 32]
      print(
          f"[!] ANOMALY: {trailing_len:,} trailing bytes detected past %%EOF."
      )
      print(f"    Hex preview: {trailing_preview.hex(' ')}")
    else:
      print("[+] Trailing Data Check: Clean.")

    print(
        f"[+] Incremental Revisions: {len(eof_positions)} marker(s) detected."
    )

  # Catalog keywords scan
  keywords_of_interest = {
      b"/OpenAction": "Automatic execution trigger",
      b"/JS": "Short JavaScript action block",
      b"/JavaScript": "Embedded JavaScript execution block",
      b"/EmbeddedFiles": "Embedded / attached files",
      b"/Launch": "External program execution command",
  }

  print("\n--- Object Keyword Scan ---")
  for kw, desc in keywords_of_interest.items():
    count = len(re.findall(re.escape(kw), data))
    if count > 0:
      print(f"  * {desc:<35} ({kw.decode()}): {count} match(es)")

Target: Grand Theft Auto VI Trailer 1_recovered.pdf (9,443,926 bytes)
------------------------------------------------------------
[!] ANOMALY: 10 trailing bytes detected past %%EOF.
    Hex preview: 00 00 00 00 00 00 00 00 00 00
[+] Incremental Revisions: 1 marker(s) detected.

--- Object Keyword Scan ---
  * Automatic execution trigger         (/OpenAction): 1 match(es)
  * Short JavaScript action block       (/JS): 1 match(es)


In [23]:
import re
import zlib

TARGET_PDF = "test_files/Grand Theft Auto VI Trailer 1_recovered.pdf"

with open(TARGET_PDF, "rb") as f:
  data = f.read()

# 1. Resolve action triggers
print("--- Locating Action Triggers ---")
open_action = re.search(rb"/OpenAction\s*(\[[^\]]+\]|<<[^>]+>>|\d+\s+\d+\s+R)", data)
if open_action:
  print(f"Found /OpenAction target: {open_action.group(1).decode('latin1')}")

# 2. Inspect individual stream objects safely
stream_pattern = re.compile(
    rb"(\d+\s+\d+\s+obj.*?<<.*?>>)\s*stream[\r\n]+(.*?)\r?\nendstream",
    re.DOTALL,
)
matches = list(stream_pattern.finditer(data))
print(f"\n--- Scanning Stream Objects (Found: {len(matches)}) ---")

for idx, match in enumerate(matches):
  header = match.group(1).decode("latin1", errors="ignore")
  payload = match.group(2)

  # Check compression filter
  is_flate = "/FlateDecode" in header
  stream_data = payload
  if is_flate:
    try:
      stream_data = zlib.decompress(payload)
    except Exception:
      pass

  # Detect binary vs text
  is_binary = b"\x00" in stream_data[:64]
  magic_preview = stream_data[:16]

  print(
      f"\nStream #{idx}: Size = {len(stream_data):,} bytes | Compressed ="
      f" {is_flate}"
  )
  dict_meta = re.findall(r"/\w+\s+[^/<>\[\]]+", header)
  print(f"  Metadata: {' '.join(dict_meta[:3])}")

  if is_binary:
    # Print safe hex preview instead of flooding terminal with raw bytes
    printable_chars = "".join(
        chr(b) if 32 <= b < 127 else "." for b in magic_preview
    )
    print(f"  [Binary Content] Hex: {magic_preview.hex(' ')}")
    print(f"  ASCII Preview:       {printable_chars}")
  else:
    # Print clean text lines (max 5 lines)
    text_snippet = stream_data.decode("latin1", errors="ignore").strip().splitlines()
    print("  [Text Content Preview]:")
    for line in text_snippet[:5]:
      print(f"    {line}")

--- Locating Action Triggers ---
Found /OpenAction target: [3 0 R/Fit]

--- Scanning Stream Objects (Found: 42) ---

Stream #0: Size = 9,357,855 bytes | Compressed = False
  Metadata: /Length 9357855
  [Binary Content] Hex: 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
  ASCII Preview:       ................

Stream #1: Size = 7,480 bytes | Compressed = True
  Metadata: /Names 60 0 R /Outlines 61 0 R /Pages 71 0 R
  [Text Content Preview]:
    q 1 0 0 1 72 769.89 cm 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g 0 G 0 g q 1 0 0 1 -72 -769.89 cm q  0.09021 0.12549 0.16472 RG  0.09021 0.12549 0.16472 rg  0.3985 w  q  Q  n  Q  Q 0 G 0 g q 1 0 0 1 -72 -769.89 cm q  0.09021 0.12549 0.16472 RG  0.09021 0.12549 0.16472 rg  0.3985 w  q  q  0.082 0.31 0.471 RG 0.082 0.31 0.471 rg 0.0 841.9006 m  0.0 841.9006 m  0.0 0.0 l  25.51212 0.0 l  25.51212 841

In [27]:
import struct
import subprocess

TARGET_PDF = "test_files/Grand Theft Auto VI Trailer 1_recovered.pdf"
OUTPUT_MP4 = "test_files/carved_gta_trailer.mp4"

MP4_START = 256
MP4_END = 573338 + 8784589  # 9,357,927

with open(TARGET_PDF, "rb") as f:
  f.seek(MP4_START)
  video_bytes = f.read(MP4_END - MP4_START)

# Prepend a 256-byte 'free' atom to realign internal stco absolute offsets
padding_atom = (
    (256).to_bytes(4, byteorder="big") + b"free" + (b"\x00" * (256 - 8))
)

with open(OUTPUT_MP4, "wb") as out:
  out.write(padding_atom)
  out.write(video_bytes)

total_size = len(padding_atom) + len(video_bytes)
print(f"[+] Repaired and saved video to {OUTPUT_MP4} ({total_size:,} bytes)")

# Quick integrity check
res = subprocess.run(
    [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "format=duration:stream=codec_name,width,height",
        OUTPUT_MP4,
    ],
    capture_output=True,
    text=True,
)
print("ffprobe validation:\n" + (res.stdout.strip() or res.stderr.strip()))

[+] Repaired and saved video to test_files/carved_gta_trailer.mp4 (9,357,927 bytes)
ffprobe validation:
[STREAM]
codec_name=h264
width=640
height=360
[/STREAM]
[STREAM]
codec_name=aac
[/STREAM]
[FORMAT]
duration=90.047007
[/FORMAT]


In [28]:
import re

TARGET_PDF = "test_files/Grand Theft Auto VI Trailer 1_recovered.pdf"
OUTPUT_HTML = "test_files/carved_skip_page.html"

# Atom boundaries for the 476 KB skip box between moov and mdat
SKIP_OFFSET = 97240
SKIP_SIZE = 476098

with open(TARGET_PDF, "rb") as f:
  f.seek(SKIP_OFFSET + 8)  # Skip 8-byte atom header
  skip_payload = f.read(SKIP_SIZE - 8)

# Carve the embedded HTML document
html_start = skip_payload.find(b"<!DOCTYPE")
html_end = skip_payload.rfind(b"</html>")

if html_start != -1 and html_end != -1:
  html_bytes = skip_payload[html_start : html_end + 7]
  with open(OUTPUT_HTML, "wb") as out:
    out.write(html_bytes)
  print(f"[+] Extracted hidden HTML payload ({len(html_bytes):,} bytes)")

  html_str = html_bytes.decode("utf-8", errors="ignore")

  # Extract hidden developer comments and tokens
  print("\n--- Extracted Credentials & Tokens ---")
  comments = re.findall(r"<!--(.*?)-->", html_str, re.DOTALL)
  for c in comments:
    clean = c.strip()
    if clean:
      print(f"  * {clean}")
else:
  print("[!] Could not locate HTML payload boundaries inside 'skip' atom.")

[+] Extracted hidden HTML payload (15,156 bytes)

--- Extracted Credentials & Tokens ---
  * todo: clean up the dev auth tokens in this file before publication. not sure why we embedded them in the html, ask jason
  * flag for staging access: TARGET_GTAVI_LEAK_DASHBOARD / key: vl-cty-gta6-2026 / nothing else interesting in this page
